# Pothole Detection with RT-DETR

End-to-end pipeline for detecting potholes in road images using **RT-DETR**
(Real-Time Detection Transformer) via the [Ultralytics](https://docs.ultralytics.com/models/rtdetr/) API.

**Datasets** (both pulled via `kagglehub`):
- [`andrewmvd/pothole-detection`](https://www.kaggle.com/datasets/andrewmvd/pothole-detection) — Pascal-VOC-XML-annotated pothole images. This is what RT-DETR is actually **trained** on.
- [`atulyakumar98/pothole-detection-dataset`](https://www.kaggle.com/datasets/atulyakumar98/pothole-detection-dataset) — `normal/` vs `potholes/` image folders with **no bounding boxes**. It can't train a detector, so it's used afterwards as an independent, weakly-labeled sanity check for the trained model.

**Pipeline**
1. Install dependencies & imports
2. Download both datasets with `kagglehub`
3. Auto-detect the annotation format (YOLO `.txt`, Pascal VOC `.xml`, or an existing `data.yaml`) and normalize everything into a single YOLO-style dataset
4. Exploratory data analysis (class balance, image sizes, bounding-box visualization)
5. Build the `data.yaml` config Ultralytics expects
6. Train RT-DETR
7. Evaluate (mAP50, mAP50-95, precision/recall, confusion matrix)
8. Visualize predictions on validation images
9. Run inference on new/test images
10. Export the trained model
11. Weak image-level sanity check on the unlabeled `atulyakumar98` dataset

> **Before running:** in the notebook settings, turn on **Internet** (for `pip install`, `kagglehub` downloads, and RT-DETR pretrained weights) and set the **Accelerator** to a GPU.


## 1. Setup

In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python

import numpy as np  # linear algebra
import pandas as pd  # data processing, CSV file I/O

import os
for dirname, _, filenames in os.walk("/kaggle/input"):
    for filename in filenames:
        print(os.path.join(dirname, filename))


In [ ]:
!pip install -q ultralytics kagglehub


In [ ]:
import os
import shutil
import random
import glob
import yaml
import xml.etree.ElementTree as ET
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

import kagglehub
from ultralytics import RTDETR

random.seed(42)
np.random.seed(42)

WORK_DIR = Path("/kaggle/working")
DATASET_DIR = WORK_DIR / "pothole_dataset"   # normalized YOLO-style dataset lives here
RUNS_DIR = WORK_DIR / "runs"

WORK_DIR.mkdir(parents=True, exist_ok=True)
DATASET_DIR.mkdir(parents=True, exist_ok=True)
print("Ultralytics imported OK. Working dir:", WORK_DIR)


## 2. Download the Datasets (via kagglehub)

Two datasets, downloaded straight from Kaggle with `kagglehub` (no manual "Add Input" needed):

- **`andrewmvd/pothole-detection`** -- Pascal-VOC-XML-annotated pothole images. This is the labeled
  dataset RT-DETR is actually trained on.
- **`atulyakumar98/pothole-detection-dataset`** -- a `normal/` vs `potholes/` image-only dataset with
  no bounding boxes. It cannot train a detector, but Section 13 uses its folder labels as a weak,
  independent sanity check of the trained model.

The cell below prints a shallow tree of each downloaded dataset so you can sanity-check the layout
before the auto-detection logic runs.

In [ ]:
def list_tree(root, max_depth=3, max_entries=6):
    root = Path(root)
    for dirpath, dirnames, filenames in os.walk(root):
        depth = len(Path(dirpath).relative_to(root).parts)
        if depth > max_depth:
            dirnames[:] = []
            continue
        indent = "  " * depth
        print(f"{indent}{Path(dirpath).name}/")
        for f in sorted(filenames)[:max_entries]:
            print(f"{indent}  {f}")
        if len(filenames) > max_entries:
            print(f"{indent}  ... (+{len(filenames) - max_entries} more files)")

detection_dataset_path = Path(kagglehub.dataset_download("andrewmvd/pothole-detection"))
demo_dataset_path = Path(kagglehub.dataset_download("atulyakumar98/pothole-detection-dataset"))

print("=== andrewmvd/pothole-detection (training) ===")
list_tree(detection_dataset_path)
print("\n=== atulyakumar98/pothole-detection-dataset (weak-label demo) ===")
list_tree(demo_dataset_path)

dataset_root = detection_dataset_path
print(f"\nTraining on dataset_root = {dataset_root}")


## 3. Auto-Detect Annotation Format

Pothole datasets on Kaggle typically show up in one of three shapes:

- **`yolo_ready`** -- a `data.yaml`/`data.yml` already ships with the dataset (train/val paths + class names).
- **`yolo_txt`** -- plain YOLO format: images plus a `labels/` folder of matching `.txt` files (one row per box: `class x_center y_center width height`, normalized).
- **`voc_xml`** -- Pascal VOC format: images plus per-image `.xml` annotation files with `<bndbox>` tags.

The cell below scans the dataset and decides which case applies.

In [ ]:
IMG_EXTS = (".jpg", ".jpeg", ".png", ".bmp")

image_files = [p for p in dataset_root.rglob("*") if p.suffix.lower() in IMG_EXTS]
xml_files = list(dataset_root.rglob("*.xml"))
yaml_files = list(dataset_root.rglob("*.yaml")) + list(dataset_root.rglob("*.yml"))
txt_label_files = [
    p for p in dataset_root.rglob("*.txt")
    if "label" in p.parent.name.lower()
]

print(f"Images found:            {len(image_files)}")
print(f"YAML config files found: {len(yaml_files)}")
print(f"YOLO .txt label files:   {len(txt_label_files)}")
print(f"VOC .xml annotations:    {len(xml_files)}")

if yaml_files:
    ANNOTATION_FORMAT = "yolo_ready"
elif txt_label_files:
    ANNOTATION_FORMAT = "yolo_txt"
elif xml_files:
    ANNOTATION_FORMAT = "voc_xml"
else:
    raise RuntimeError(
        "Could not auto-detect the annotation format. Inspect `dataset_root` manually "
        "(see the tree printed above) and adapt Section 3/4 to your dataset's layout."
    )

print(f"\nDetected annotation format: {ANNOTATION_FORMAT}")


## 4. Normalize the Dataset into a Single YOLO-Style Layout

Whatever format we detected, we convert/copy everything into one canonical structure under
`/kaggle/working/pothole_dataset/{train,valid,test}/{images,labels}` so the rest of the notebook
does not need to care which branch produced it.

In [ ]:
def voc_bbox_to_yolo(img_w, img_h, xmin, xmax, ymin, ymax):
    dw, dh = 1.0 / img_w, 1.0 / img_h
    x = (xmin + xmax) / 2.0 * dw
    y = (ymin + ymax) / 2.0 * dh
    w = (xmax - xmin) * dw
    h = (ymax - ymin) * dh
    return x, y, w, h


def find_image_for(filename, search_root):
    matches = list(search_root.rglob(filename))
    return matches[0] if matches else None


def convert_voc_to_yolo(xml_files, search_root, flat_images_dir, flat_labels_dir, classes):
    # Convert every VOC XML file to a YOLO .txt label and copy its image. Returns updated class list.
    flat_images_dir.mkdir(parents=True, exist_ok=True)
    flat_labels_dir.mkdir(parents=True, exist_ok=True)

    for xml_path in xml_files:
        tree = ET.parse(xml_path)
        root = tree.getroot()

        filename_node = root.find("filename")
        filename = filename_node.text.strip() if filename_node is not None else xml_path.stem + ".jpg"
        img_path = find_image_for(filename, search_root) or find_image_for(xml_path.stem + ".jpg", search_root)
        if img_path is None:
            continue

        size_node = root.find("size")
        if size_node is not None:
            img_w = int(size_node.find("width").text)
            img_h = int(size_node.find("height").text)
        else:
            img = cv2.imread(str(img_path))
            img_h, img_w = img.shape[:2]

        lines = []
        for obj in root.findall("object"):
            cls_name = obj.find("name").text.strip().lower()
            if cls_name not in classes:
                classes.append(cls_name)
            cls_id = classes.index(cls_name)

            bnd = obj.find("bndbox")
            xmin, xmax = float(bnd.find("xmin").text), float(bnd.find("xmax").text)
            ymin, ymax = float(bnd.find("ymin").text), float(bnd.find("ymax").text)

            # Clip to image bounds and drop degenerate (zero/negative-area) boxes -- both are
            # annotation errors that occur often enough in scraped VOC datasets, and either one
            # can produce NaN gradients in the GIoU loss during training.
            xmin, xmax = max(0.0, min(xmin, xmax)), min(img_w, max(xmin, xmax))
            ymin, ymax = max(0.0, min(ymin, ymax)), min(img_h, max(ymin, ymax))
            if xmax - xmin < 1 or ymax - ymin < 1:
                continue

            x, y, w, h = voc_bbox_to_yolo(img_w, img_h, xmin, xmax, ymin, ymax)
            lines.append(f"{cls_id} {x:.6f} {y:.6f} {w:.6f} {h:.6f}")

        shutil.copy(img_path, flat_images_dir / img_path.name)
        (flat_labels_dir / (Path(filename).stem + ".txt")).write_text("\n".join(lines))

    return classes


def split_pairs(image_label_pairs, out_root, ratios=(0.8, 0.1, 0.1)):
    pairs = list(image_label_pairs)
    random.shuffle(pairs)
    n = len(pairs)
    n_train = int(n * ratios[0])
    n_val = int(n * ratios[1])
    splits = {
        "train": pairs[:n_train],
        "valid": pairs[n_train:n_train + n_val],
        "test": pairs[n_train + n_val:],
    }
    for split, split_pairs_ in splits.items():
        (out_root / split / "images").mkdir(parents=True, exist_ok=True)
        (out_root / split / "labels").mkdir(parents=True, exist_ok=True)
        for img_path, lbl_path in split_pairs_:
            shutil.copy(img_path, out_root / split / "images" / img_path.name)
            dest_lbl = out_root / split / "labels" / (img_path.stem + ".txt")
            if lbl_path is not None and Path(lbl_path).exists():
                shutil.copy(lbl_path, dest_lbl)
            else:
                dest_lbl.touch()  # image with no detections -> empty label file
        print(f"  {split:5s}: {len(split_pairs_)} images")
    return out_root


def find_existing_split_dirs(root):
    # Detect dataset that already ships pre-split train/valid(/val)/test folders with images+labels.
    found = {}
    for split_name in ("train", "valid", "val", "test"):
        for d in root.rglob(split_name):
            if d.is_dir() and (d / "images").is_dir():
                key = "valid" if split_name == "val" else split_name
                found[key] = d
    return found


In [ ]:
class_names = ["pothole"]  # sensible default; extended automatically for multi-class VOC datasets

if ANNOTATION_FORMAT == "yolo_ready":
    src_yaml_path = yaml_files[0]
    with open(src_yaml_path) as f:
        src_yaml = yaml.safe_load(f)
    print("Found existing data.yaml:", src_yaml_path)
    print(src_yaml)

    class_names = src_yaml.get("names", class_names)
    if isinstance(class_names, dict):
        class_names = [class_names[k] for k in sorted(class_names)]

    # Dataset yaml paths are often relative to wherever it was originally built (e.g. Colab/local
    # paths) and break once re-attached on Kaggle, so we re-point train/val/test at whatever split
    # folders actually exist on disk next to the yaml file instead of trusting the stored paths.
    existing_splits = find_existing_split_dirs(dataset_root)
    assert existing_splits, "data.yaml found but no train/valid/test image folders could be located."

    for split, split_dir in existing_splits.items():
        dest = DATASET_DIR / split
        if dest.exists():
            shutil.rmtree(dest)
        shutil.copytree(split_dir, dest)
        print(f"  {split:5s}: {len(list((dest / 'images').iterdir()))} images (copied from {split_dir})")

elif ANNOTATION_FORMAT == "yolo_txt":
    existing_splits = find_existing_split_dirs(dataset_root)
    if existing_splits:
        print("Dataset already split into:", list(existing_splits.keys()))
        for split, split_dir in existing_splits.items():
            dest = DATASET_DIR / split
            if dest.exists():
                shutil.rmtree(dest)
            shutil.copytree(split_dir, dest)
            print(f"  {split:5s}: {len(list((dest / 'images').iterdir()))} images (copied from {split_dir})")
    else:
        print("No pre-existing split found -- building an 80/10/10 train/valid/test split.")
        pairs = []
        for img_path in image_files:
            lbl_path = None
            for cand in dataset_root.rglob(img_path.stem + ".txt"):
                if "label" in cand.parent.name.lower():
                    lbl_path = cand
                    break
            pairs.append((img_path, lbl_path))
        split_pairs(pairs, DATASET_DIR)

    # Pick up class names from a classes.txt/names file if the dataset ships one.
    names_files = list(dataset_root.rglob("classes.txt")) + list(dataset_root.rglob("*.names"))
    if names_files:
        class_names = [l.strip() for l in names_files[0].read_text().splitlines() if l.strip()]

elif ANNOTATION_FORMAT == "voc_xml":
    print("Converting Pascal VOC XML annotations to YOLO format...")
    flat_images = WORK_DIR / "_flat_voc" / "images"
    flat_labels = WORK_DIR / "_flat_voc" / "labels"
    class_names = convert_voc_to_yolo(xml_files, dataset_root, flat_images, flat_labels, [])
    if not class_names:
        class_names = ["pothole"]
    print("Classes discovered from XML:", class_names)

    pairs = []
    for img_path in sorted(flat_images.iterdir()):
        lbl_path = flat_labels / (img_path.stem + ".txt")
        pairs.append((img_path, lbl_path if lbl_path.exists() else None))
    split_pairs(pairs, DATASET_DIR)

print(f"\nFinal class names: {class_names}")
print(f"Normalized dataset ready at: {DATASET_DIR}")


## 5. Exploratory Data Analysis

In [ ]:
def yolo_label_path(img_path):
    return img_path.parent.parent / "labels" / (img_path.stem + ".txt")


split_counts = {}
box_counts = []
img_sizes = []

for split in ("train", "valid", "test"):
    split_dir = DATASET_DIR / split / "images"
    if not split_dir.exists():
        continue
    imgs = list(split_dir.iterdir())
    split_counts[split] = len(imgs)
    for img_path in imgs:
        lbl_path = yolo_label_path(img_path)
        n_boxes = 0
        if lbl_path.exists():
            n_boxes = sum(1 for line in lbl_path.read_text().splitlines() if line.strip())
        box_counts.append(n_boxes)
        img = cv2.imread(str(img_path))
        if img is not None:
            img_sizes.append((img.shape[1], img.shape[0]))  # (w, h)

print("Images per split:", split_counts)
print(f"Total images: {sum(split_counts.values())}")
print(f"Images with zero pothole boxes: {sum(1 for c in box_counts if c == 0)}")
print(f"Average boxes per image: {np.mean(box_counts):.2f}")


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

axes[0].bar(split_counts.keys(), split_counts.values(), color="steelblue")
axes[0].set_title("Images per split")
axes[0].set_ylabel("count")

axes[1].hist(box_counts, bins=range(0, max(box_counts) + 2), color="indianred", edgecolor="black")
axes[1].set_title("Potholes per image")
axes[1].set_xlabel("# bounding boxes")

if img_sizes:
    widths, heights = zip(*img_sizes)
    axes[2].scatter(widths, heights, alpha=0.3, s=10)
    axes[2].set_title("Image resolution distribution")
    axes[2].set_xlabel("width (px)")
    axes[2].set_ylabel("height (px)")

plt.tight_layout()
plt.show()


In [ ]:
def draw_yolo_boxes(img_path, label_path, class_names):
    img = cv2.imread(str(img_path))
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    h, w = img.shape[:2]
    if label_path.exists():
        for line in label_path.read_text().splitlines():
            parts = line.split()
            if len(parts) != 5:
                continue
            cls_id, xc, yc, bw, bh = int(parts[0]), *map(float, parts[1:])
            x1 = int((xc - bw / 2) * w)
            y1 = int((yc - bh / 2) * h)
            x2 = int((xc + bw / 2) * w)
            y2 = int((yc + bh / 2) * h)
            cv2.rectangle(img, (x1, y1), (x2, y2), (255, 0, 0), 2)
            label = class_names[cls_id] if cls_id < len(class_names) else str(cls_id)
            cv2.putText(img, label, (x1, max(y1 - 5, 0)), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 0, 0), 2)
    return img


sample_dir = DATASET_DIR / "train" / "images"
sample_imgs = random.sample(list(sample_dir.iterdir()), k=min(9, len(list(sample_dir.iterdir()))))

fig, axes = plt.subplots(3, 3, figsize=(13, 13))
for ax, img_path in zip(axes.flat, sample_imgs):
    lbl_path = yolo_label_path(img_path)
    ax.imshow(draw_yolo_boxes(img_path, lbl_path, class_names))
    ax.set_title(img_path.name, fontsize=8)
    ax.axis("off")
plt.tight_layout()
plt.show()


## 6. Build `data.yaml` for Ultralytics

In [ ]:
data_yaml_path = WORK_DIR / "data.yaml"

data_yaml = {
    "path": str(DATASET_DIR),
    "train": "train/images",
    "val": "valid/images",
    "nc": len(class_names),
    "names": class_names,
}

if (DATASET_DIR / "test" / "images").exists() and any((DATASET_DIR / "test" / "images").iterdir()):
    data_yaml["test"] = "test/images"

with open(data_yaml_path, "w") as f:
    yaml.safe_dump(data_yaml, f, sort_keys=False)

print(f"Wrote {data_yaml_path}:\n")
print(data_yaml_path.read_text())


## 7. Train RT-DETR

`rtdetr-l.pt` (large) is a good accuracy/speed default; swap for `rtdetr-x.pt` for higher accuracy
(slower) or fine-tune epochs/batch size to fit your GPU budget.

**`amp=False`:** RT-DETR's transformer decoder + GIoU loss is known to occasionally diverge to NaN
under automatic mixed precision partway through training (loss goes to `nan`, val mAP collapses to
0 and never recovers). Training in full precision is slightly slower but avoids this; if you hit NaN
losses anyway, also try lowering `lr0` (e.g. `lr0=0.0001`) instead of `optimizer="auto"`.

In [ ]:
EPOCHS = 60
IMG_SIZE = 640
BATCH = 8
MODEL_VARIANT = "rtdetr-l.pt"  # or "rtdetr-x.pt"

model = RTDETR(MODEL_VARIANT)

train_results = model.train(
    data=str(data_yaml_path),
    epochs=EPOCHS,
    imgsz=IMG_SIZE,
    batch=BATCH,
    project=str(RUNS_DIR),
    name="pothole_rtdetr",
    seed=42,
    patience=15,
    amp=False,   # avoid RT-DETR's known NaN-loss divergence under mixed precision
    plots=True,
)

run_dir = Path(train_results.save_dir)
print(f"\nTraining finished. Run artifacts saved to: {run_dir}")


## 8. Training Curves & Evaluation Metrics

In [ ]:
for fname in ("results.png", "confusion_matrix.png", "confusion_matrix_normalized.png", "PR_curve.png"):
    fpath = run_dir / fname
    if fpath.exists():
        plt.figure(figsize=(9, 6))
        plt.imshow(mpimg.imread(fpath))
        plt.axis("off")
        plt.title(fname)
        plt.show()


In [ ]:
best_weights = run_dir / "weights" / "best.pt"
best_model = RTDETR(str(best_weights))

metrics = best_model.val(data=str(data_yaml_path), split="val")

print(f"mAP50:      {metrics.box.map50:.4f}")
print(f"mAP50-95:   {metrics.box.map:.4f}")
print(f"Precision:  {metrics.box.mp:.4f}")
print(f"Recall:     {metrics.box.mr:.4f}")


## 9. Visualize Predictions on Validation Images

In [ ]:
val_images_dir = DATASET_DIR / "valid" / "images"
val_samples = random.sample(list(val_images_dir.iterdir()), k=min(6, len(list(val_images_dir.iterdir()))))

pred_results = best_model.predict(source=[str(p) for p in val_samples], conf=0.25, imgsz=IMG_SIZE)

fig, axes = plt.subplots(2, 3, figsize=(15, 10))
for ax, result in zip(axes.flat, pred_results):
    annotated = result.plot()  # BGR numpy array with boxes drawn
    ax.imshow(cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB))
    ax.set_title(Path(result.path).name, fontsize=9)
    ax.axis("off")
plt.tight_layout()
plt.show()


## 10. Inference on New / Test Images

Point `test_source` at any image, folder of images, or (if present) the held-out `test` split.

In [ ]:
test_source = DATASET_DIR / "test" / "images"
if not test_source.exists() or not any(test_source.iterdir()):
    test_source = val_images_dir  # fall back to validation images if no test split exists

inference_out_dir = WORK_DIR / "inference_results"

results = best_model.predict(
    source=str(test_source),
    conf=0.25,
    imgsz=IMG_SIZE,
    save=True,
    project=str(WORK_DIR),
    name="inference_results",
    exist_ok=True,
)

for r in results[:5]:
    n_detections = len(r.boxes)
    print(f"{Path(r.path).name}: {n_detections} pothole(s) detected")

print(f"\nAnnotated images saved under: {inference_out_dir}")


## 11. Export the Trained Model

In [ ]:
try:
    exported_path = best_model.export(format="onnx")
    print(f"Exported ONNX model to: {exported_path}")
except Exception as e:
    print(f"Export skipped/failed: {e}")


## 12. Save Final Artifacts

In [ ]:
final_weights_path = WORK_DIR / "pothole_rtdetr_best.pt"
shutil.copy(best_weights, final_weights_path)

print("Summary")
print("=" * 40)
print(f"Classes:        {class_names}")
print(f"Best weights:   {final_weights_path}")
print(f"mAP50:          {metrics.box.map50:.4f}")
print(f"mAP50-95:       {metrics.box.map:.4f}")
print(f"Training run:   {run_dir}")


## 13. Weak Image-Level Sanity Check on the atulyakumar98 Dataset

`atulyakumar98/pothole-detection-dataset` has no bounding boxes, but its folder names (`potholes/`
vs `normal/`) give a weak, image-level ground truth: a good detector should fire at least one box on
most `potholes/` images and stay silent on most `normal/` images. This is **not** a substitute for
the mAP computed in Section 8 -- it is an out-of-distribution smoke test on data the model never
trained on.

In [ ]:
potholes_dir = next((p for p in demo_dataset_path.rglob("*") if p.is_dir() and p.name.lower() == "potholes"), None)
normal_dir = next((p for p in demo_dataset_path.rglob("*") if p.is_dir() and p.name.lower() == "normal"), None)
assert potholes_dir and normal_dir, "Expected 'potholes' and 'normal' folders in the atulyakumar98 dataset."


def images_in(d):
    return [p for p in d.iterdir() if p.suffix.lower() in IMG_EXTS]


potholes_imgs = images_in(potholes_dir)
normal_imgs = images_in(normal_dir)
print(f"potholes/: {len(potholes_imgs)} images | normal/: {len(normal_imgs)} images")


def detected_any(img_path, conf=0.25):
    r = best_model.predict(source=str(img_path), conf=conf, imgsz=IMG_SIZE, verbose=False)[0]
    return len(r.boxes) > 0


tp = sum(detected_any(p) for p in potholes_imgs)
fn = len(potholes_imgs) - tp
fp = sum(detected_any(p) for p in normal_imgs)
tn = len(normal_imgs) - fp

accuracy = (tp + tn) / (tp + tn + fp + fn)
recall = tp / len(potholes_imgs) if potholes_imgs else float("nan")
specificity = tn / len(normal_imgs) if normal_imgs else float("nan")

print(f"Image-level recall on potholes/ images:    {recall:.2%}  ({tp}/{len(potholes_imgs)} had >=1 detection)")
print(f"Image-level specificity on normal/ images: {specificity:.2%}  ({tn}/{len(normal_imgs)} had 0 detections)")
print(f"Overall weak-label accuracy:               {accuracy:.2%}")


In [ ]:
sample_demo = (
    random.sample(potholes_imgs, k=min(3, len(potholes_imgs)))
    + random.sample(normal_imgs, k=min(3, len(normal_imgs)))
)
demo_results = best_model.predict(source=[str(p) for p in sample_demo], conf=0.25, imgsz=IMG_SIZE)

fig, axes = plt.subplots(2, 3, figsize=(15, 10))
for ax, result in zip(axes.flat, demo_results):
    annotated = result.plot()
    src_path = Path(result.path)
    ax.imshow(cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB))
    ax.set_title(f"{src_path.parent.name}/{src_path.name}", fontsize=9)
    ax.axis("off")
plt.tight_layout()
plt.show()
